In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

import tf2onnx
import onnx
import os
import cv2

In [ ]:
# print(keras.__version__)


In [ ]:
def zoom_out(image, zoom_factor=0.7, border_type=cv2.BORDER_CONSTANT):
    """
    Zooms out of an image while keeping the original border dimensions.

    :param image: Input OpenCV image matrix (BGR).
    :param zoom_factor: Multiplier < 1.0 to shrink the image.
    :param border_type: OpenCV border style (e.g., cv2.BORDER_CONSTANT, cv2.BORDER_REPLICATE, cv2.BORDER_REFLECT).
    :param border_value: BGR tuple color if using cv2.BORDER_CONSTANT.
    """
    # 1. Get original dimensions
    h, w = image.shape[:2]

    # 2. Calculate the new shrunken dimensions
    new_w = int(w * zoom_factor)
    new_h = int(h * zoom_factor)

    # 3. Resize the image down
    shrunken_img = cv2.resize(image, (new_w, new_h))
            
    # 4. Calculate required padding to reach original dimensions
    top = (h - new_h) // 2
    bottom = h - new_h - top
    left = (w - new_w) // 2
    right = w - new_w - left

    # 5. Re-add borders to match original sizes
    zoomed_out_img = cv2.copyMakeBorder(
        shrunken_img, 
        top, bottom, left, right, 
        borderType=border_type, 
        value=(255,255,255)
    )

    return zoomed_out_img


In [ ]:
def zoom_in(image, zoom_factor=1.2):
    if zoom_factor <= 1.0:
        return image
        
    height, width = image.shape[:2]
    
    # Calculate the new dimensions of the cropped area
    new_h, new_w = int(height / zoom_factor), int(width / zoom_factor)
    
    # Determine the boundaries for the center crop
    start_y = (height - new_h) // 2
    start_x = (width - new_w) // 2
    end_y = start_y + new_h
    end_x = start_x + new_w
    
    # Crop and upscale back to the original size
    cropped = image[start_y:end_y, start_x:end_x]
    return cv2.resize(cropped, (width, height), interpolation=cv2.INTER_LINEAR)



In [ ]:
#load testing data
classes = ['1','2','3','4','5','6','7','8','9']

train_images = []
train_labels = []
test_images = []
test_labels = []

training_folder = "db/data/training_data"
testing_folder = "db/data/testing_data"

for num_class in classes:
    folder_name = f"{os.getcwd()}/{training_folder}/{num_class}"
    for index , filename in enumerate(os.listdir(folder_name)):
        # if index >= 500:
        #     break
        if filename.endswith((".png", "jpg", "jpeg")):
            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)
            img =cv2.imread(os.path.join(folder_name,filename), cv2.IMREAD_GRAYSCALE)
            resized_img = cv2.resize(img,(28,28))
            retval, thresh_img = cv2.threshold(resized_img, 180, 255, cv2.THRESH_OTSU)
            # thresh_img = resized_img
            thresh_12 = zoom_in(thresh_img)
            thresh_06 = zoom_out(thresh_img,zoom_factor=0.6)
            thresh_05 = zoom_out(thresh_img,zoom_factor=0.5)
            thresh_04 = zoom_out(thresh_img,zoom_factor=0.4)
            thresh_03 = zoom_out(thresh_img,zoom_factor=0.3)
            thresh_02 = zoom_out(thresh_img,zoom_factor=0.2)
            train_images.append(thresh_img)
            train_images.append(thresh_12)
            train_images.append(thresh_06)
            train_images.append(thresh_05)
            train_images.append(thresh_04)
            train_images.append(thresh_03)
            train_images.append(thresh_02)
            train_images.append(255-thresh_img)
            train_images.append(255-thresh_12)
            train_images.append(255-thresh_06)
            train_images.append(255-thresh_05)
            train_images.append(255-thresh_04)
            train_images.append(255-thresh_03)
            train_images.append(255-thresh_02)

            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)   
            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)
            train_labels.append(int(num_class)-1)      


for num_class in classes:
    folder_name = f"{os.getcwd()}/{testing_folder}/{num_class}"
    for index , filename in enumerate(os.listdir(folder_name)):
        if filename.endswith((".png", "jpg", "jpeg")):
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            img =cv2.imread(os.path.join(folder_name,filename), cv2.IMREAD_GRAYSCALE)
            resized_img = cv2.resize(img,(28,28))
            retval, thresh_img = cv2.threshold(resized_img, 180, 255, cv2.THRESH_OTSU)
            thresh_12 = zoom_in(thresh_img)
            thresh_06 = zoom_out(thresh_img,zoom_factor=0.6)
            thresh_05 = zoom_out(thresh_img,zoom_factor=0.5)
            thresh_04 = zoom_out(thresh_img,zoom_factor=0.4)
            thresh_03 = zoom_out(thresh_img,zoom_factor=0.3)
            thresh_02 = zoom_out(thresh_img,zoom_factor=0.2)
        
            test_images.append(thresh_img)
            test_images.append(255-thresh_12)
            test_images.append(255-thresh_img)
            test_images.append(255-thresh_06)
            test_images.append(255-thresh_05)
            test_images.append(255-thresh_04)
            test_images.append(255-thresh_03)
            test_images.append(255-thresh_02)

            test_images.append(thresh_12)
            test_images.append(thresh_06)
            test_images.append(thresh_05)
            test_images.append(thresh_04)
            test_images.append(thresh_03)
            test_images.append(thresh_02)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            test_labels.append(int(num_class)-1)
            


train_images = np.array(train_images,dtype=np.float32)/255.0
train_images = train_images.reshape(-1,28,28,1)
test_images = np.array(test_images,dtype=np.float32)/255.0
test_images = test_images.reshape(-1,28,28,1)
train_labels = np.array(train_labels, dtype=np.int32)
test_labels = np.array(test_labels, dtype=np.int32)




In [ ]:
plt.figure(figsize=(10,10))
num_class = 450
start = (num_class-1)*50
for i in range(start,start+50):
    plt.subplot(10,10,i-start+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(train_labels[i])
plt.show()

In [ ]:
# create cnn
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPool2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPool2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(9, activation='softmax'),
])


In [ ]:
model.summary()

In [ ]:
# model.load_weights("model.h5")

In [ ]:
model.compile(optimizer='adam',
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
            metrics=['accuracy']
)

history = model.fit(train_images,train_labels,epochs=15,validation_data=(test_images,test_labels))


In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim([0.5,1])
plt.legend(loc="lower right")

In [ ]:
dummy_input = cv2.imread("img.jpg",cv2.IMREAD_GRAYSCALE)

# Resize to exactly 28x28 pixels -> Shape becomes (28, 28)
img_resized = cv2.resize(dummy_input, (28, 28))
_,img_resized = cv2.threshold(img_resized,130,255,cv2.THRESH_OTSU)
# Normalize pixels to 0.0 - 1.0 (Most models expect this)
img_normalized = img_resized.astype('float32') / 255.0

# CRITICAL STEP 1: Add Channel Dimension -> Shape becomes (28, 28, 1)
img_with_channel = np.expand_dims(img_normalized, axis=-1)
plt.figure(figsize=(2,2))
plt.xticks([])
plt.yticks([])
plt.grid(False)
plt.imshow(img_with_channel)
plt.show()
# CRITICAL STEP 2: Add Batch Dimension -> Shape becomes (1, 28, 28, 1)
input_batch = np.expand_dims(img_with_channel, axis=0)
predictions = model.predict(input_batch)

# Get the class index with the highest probability
predicted_class = np.argmax(predictions[0])
confidence = predictions[0][predicted_class]
print(predictions[0])
print(f"\nPredicted Class: {predicted_class}")
print(f"Confidence Score: {confidence * 100:.2f}%")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 3. (Optional) Apply dynamic range quantization to compress the model size
# converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 4. Convert the model to the TFLite binary format
tflite_model = converter.convert()

# 5. Save the converted model as a .tflite file
with open('models/model3.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model successfully exported to .tflite format.")